# Week 3 · Course 4 · Model Selection and Regularization

**Sections**
1. Best Subset Selection — exhaustive search over all 2ᵖ subsets `(0:00 – 0:20)`
2. Stepwise Selection — forward (AIC) and backward (p-value) `(0:20 – 0:40)`
3. Choosing the Optimal Model Size — Cp, AIC, BIC, Adjusted R² `(0:40 – 1:00)`
4. Validation & Cross-Validation for Model Selection — one-SE rule `(1:00 – 1:15)`
5. Ridge Regression — L2 shrinkage, coefficient paths, bias–variance `(1:15 – 1:35)`
6. The Lasso — L1 sparsity, variable selection, geometric picture `(1:35 – 1:55)`
7. Dimension Reduction — PCR (unsupervised) and PLS (supervised) `(1:55 – 2:10)`

**Libraries introduced:** `sklearn.linear_model.Ridge`, `RidgeCV`, `Lasso`, `LassoCV`,
`sklearn.cross_decomposition.PLSRegression`, `sklearn.decomposition.PCA`,
`sklearn.preprocessing.StandardScaler`, `sklearn.model_selection.cross_val_score`,
`statsmodels` (`.aic`, `.bic`, `.rsquared_adj`), `itertools.combinations`

## Setup

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

# scikit-learn
from sklearn.linear_model import (
    LinearRegression, Ridge, RidgeCV, Lasso, LassoCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.datasets import load_diabetes, fetch_california_housing

# statsmodels — for inference and information criteria
import statsmodels.api as sm
import statsmodels.formula.api as smf

# scipy
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

### Load datasets

We use three datasets throughout this notebook:
- **Credit** (ISLR) — 400 customers, predict `balance` from 10 predictors
- **Diabetes** (sklearn) — 442 patients, 10 features, predict disease progression
- **California Housing** (sklearn) — 20 640 districts, 8 features, predict house value

In [ ]:
# ── Credit dataset (ISLR) ────────────────────────────────────────────────────
credit_url = "https://www.statlearning.com/s/Credit.csv"
try:
    credit = pd.read_csv(credit_url, index_col=0)
    # Standardise column names
    credit.columns = [c.strip() for c in credit.columns]
except Exception:
    # Reproducible fallback
    rng = np.random.default_rng(0)
    N = 400
    income   = rng.uniform(10, 180, N)
    limit    = 1000 + 5 * income + rng.normal(0, 500, N)
    rating   = limit / 5 + rng.normal(0, 20, N)
    cards    = rng.integers(1, 9, N).astype(float)
    age      = rng.uniform(23, 98, N)
    education= rng.integers(5, 20, N).astype(float)
    gender   = rng.choice(['Male', 'Female'], N)
    student  = rng.choice(['No', 'Yes'], N, p=[0.7, 0.3])
    married  = rng.choice(['No', 'Yes'], N, p=[0.4, 0.6])
    ethnicity= rng.choice(['African American', 'Asian', 'Caucasian'], N)
    balance  = np.clip(
        0.8*income + 0.05*limit + 19*(gender=='Female') +
        400*(student=='Yes') + rng.normal(0, 200, N), 0, None
    )
    credit = pd.DataFrame(dict(
        Income=income, Limit=limit, Rating=rating, Cards=cards,
        Age=age, Education=education, Gender=gender, Student=student,
        Married=married, Ethnicity=ethnicity, Balance=balance
    ))

print(f"Credit dataset: {credit.shape}")
credit.head()

In [ ]:
# ── Diabetes dataset ─────────────────────────────────────────────────────────
diab_raw = load_diabetes(as_frame=True)
diab = diab_raw.frame
X_diab = diab_raw.data.values
y_diab = diab_raw.target.values
diab_feature_names = diab_raw.feature_names
print(f"Diabetes dataset: {diab.shape}")
print("Features:", diab_feature_names)

# ── California Housing ───────────────────────────────────────────────────────
cal_raw = fetch_california_housing(as_frame=True)
cal = cal_raw.frame
print(f"California Housing: {cal.shape}")

---
## 1. Best Subset Selection

**Idea:** for every possible subset of predictors, fit OLS and record the RSS. For each size k, keep the best model M_k. Then choose the best k using an external criterion.

**Complexity:** 2^p models total. For p=10: 1024. For p=30: ~10^9 — intractable.

### 1.1 Exhaustive search — numeric predictors in Credit

In [ ]:
# Use numeric predictors only for clean demonstration
num_cols = ['Income', 'Limit', 'Rating', 'Cards', 'Age', 'Education']
X_cr = credit[num_cols].values
y_cr = credit['Balance'].values
p_cr = len(num_cols)
n_cr = len(y_cr)

# Best subset search
best_subsets = {}   # k -> {'rss': float, 'r2': float, 'features': list}
all_results  = []   # all models for the dot-cloud plot

for k in range(1, p_cr + 1):
    best_rss = np.inf
    for combo in combinations(range(p_cr), k):
        Xk = X_cr[:, combo]
        lr = LinearRegression().fit(Xk, y_cr)
        y_hat = lr.predict(Xk)
        rss = np.sum((y_cr - y_hat) ** 2)
        tss = np.sum((y_cr - y_cr.mean()) ** 2)
        r2  = 1 - rss / tss
        all_results.append({'k': k, 'rss': rss, 'r2': r2, 'features': combo})
        if rss < best_rss:
            best_rss = rss
            best_subsets[k] = {'rss': rss, 'r2': r2,
                               'features': [num_cols[i] for i in combo]}

res_df = pd.DataFrame(all_results)
print("Best subset models per size:")
for k, v in best_subsets.items():
    print(f"  k={k}: {v['features']}  RSS={v['rss']:.0f}  R²={v['r2']:.4f}")

### 1.2 Best-subset frontier plot — RSS and R²

In [ ]:
ks_front  = list(best_subsets.keys())
rss_front = [best_subsets[k]['rss'] for k in ks_front]
r2_front  = [best_subsets[k]['r2']  for k in ks_front]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# RSS panel
ax1.scatter(res_df['k'], res_df['rss'], alpha=0.35, s=18, color='steelblue', label='all models')
ax1.plot(ks_front, rss_front, 'ro-', ms=9, lw=2, label='best (frontier)', zorder=5)
ax1.set_xlabel('Number of Predictors'); ax1.set_ylabel('RSS')
ax1.set_title('Best Subset — RSS'); ax1.legend()

# R² panel
ax2.scatter(res_df['k'], res_df['r2'], alpha=0.35, s=18, color='steelblue')
ax2.plot(ks_front, r2_front, 'ro-', ms=9, lw=2, label='best (frontier)', zorder=5)
ax2.set_xlabel('Number of Predictors'); ax2.set_ylabel('R²')
ax2.set_title('Best Subset — R²'); ax2.legend()

plt.suptitle('Best Subset Selection: Credit data (6 numeric predictors)', y=1.02)
plt.tight_layout()
plt.show()

print("\nObservation: RSS always decreases, R² always increases as k grows.")
print("We cannot use these alone to pick k — they always prefer the full model.")

---
## 2. Stepwise Selection

For large p (e.g. p=100), exhaustive search is infeasible. Stepwise methods search only **1 + p(p+1)/2** models.

### 2.1 Forward stepwise by AIC

In [ ]:
def forward_selection_aic(df: pd.DataFrame, target: str) -> list[str]:
    """Greedy forward selection using AIC as the criterion."""
    remaining = set(df.select_dtypes(include='number').columns) - {target}
    selected = []
    current_aic = smf.ols(f'{target} ~ 1', data=df).fit().aic
    print(f"Start  | AIC={current_aic:.2f} | predictors=[]")
    while remaining:
        aics = {}
        for var in remaining:
            formula = f"{target} ~ {' + '.join(selected + [var])}"
            aics[var] = smf.ols(formula, data=df).fit().aic
        best_var = min(aics, key=aics.get)
        if aics[best_var] < current_aic:
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = aics[best_var]
            print(f"Add {best_var:<12} | AIC={current_aic:.2f} | predictors={selected}")
        else:
            print(f"Stop   | adding any variable increases AIC.")
            break
    return selected

print("=== Forward stepwise (AIC) on Credit data ===")
fwd_vars = forward_selection_aic(credit[num_cols + ['Balance']], 'Balance')

### 2.2 Backward elimination by p-value

In [ ]:
def backward_elimination(df: pd.DataFrame, target: str, alpha: float = 0.05) -> list[str]:
    """Remove predictors with p-value above alpha, one at a time."""
    preds = [c for c in df.select_dtypes(include='number').columns if c != target]
    while preds:
        formula = f"{target} ~ {' + '.join(preds)}"
        result = smf.ols(formula, data=df).fit()
        pvals = result.pvalues.drop('Intercept', errors='ignore')
        max_p = pvals.max()
        if max_p > alpha:
            drop = pvals.idxmax()
            preds.remove(drop)
            print(f"Remove {drop:<12} | p={max_p:.4f} | remaining={preds}")
        else:
            print(f"Stop   | all p < {alpha} | final model: {preds}")
            break
    return preds

print("=== Backward elimination (α=0.05) on Credit data ===")
bwd_vars = backward_elimination(credit[num_cols + ['Balance']], 'Balance')

print(f"\nForward selected: {fwd_vars}")
print(f"Backward selected: {bwd_vars}")

### 2.3 Best subset vs. forward — comparison table (ISLR Table 6.1 style)

In [ ]:
# Reproduce comparison for first 4 sizes
fwd_sequence = []
remaining_fwd = list(num_cols)
selected_fwd = []
for _ in range(min(4, p_cr)):
    best_rss_step, best_var = np.inf, None
    for var in remaining_fwd:
        Xk = X_cr[:, [num_cols.index(v) for v in selected_fwd + [var]]]
        lr = LinearRegression().fit(Xk, y_cr)
        rss = np.sum((y_cr - lr.predict(Xk))**2)
        if rss < best_rss_step:
            best_rss_step, best_var = rss, var
    selected_fwd.append(best_var)
    remaining_fwd.remove(best_var)
    fwd_sequence.append(list(selected_fwd))

comparison = pd.DataFrame({
    '# Vars': range(1, 5),
    'Best Subset': [str(best_subsets[k]['features']) for k in range(1, 5)],
    'Forward Stepwise': [str(s) for s in fwd_sequence],
})
print(comparison.to_string(index=False))

---
## 3. Choosing Optimal Model Size — Cp, AIC, BIC, Adjusted R²

Training RSS and R² always improve as we add predictors — they are not valid for comparing models of different sizes. We need criteria that **penalise complexity**.

| Criterion | Formula | Goal |
|-----------|---------|------|
| Mallow's Cp | (RSS + 2dσ̂²) / n | minimise |
| AIC | −2 log L + 2d | minimise |
| BIC | (RSS + log(n)·d·σ̂²) / n | minimise |
| Adjusted R² | 1 − RSS/(n−d−1) / TSS/(n−1) | maximise |

### 3.1 Compute all four criteria for each best-subset model

In [ ]:
# Estimate σ² from the full model
m_full = sm.OLS(y_cr, sm.add_constant(X_cr)).fit()
sigma2_hat = m_full.mse_resid   # RSS / (n - p - 1)
tss = np.sum((y_cr - y_cr.mean())**2)

rows = []
for k in range(1, p_cr + 1):
    idx = [num_cols.index(f) for f in best_subsets[k]['features']]
    Xk = sm.add_constant(X_cr[:, idx])
    mk = sm.OLS(y_cr, Xk).fit()
    rss = mk.ssr
    d   = k + 1  # predictors + intercept
    cp  = (rss + 2 * k * sigma2_hat) / n_cr
    rows.append({
        'k': k,
        'features': best_subsets[k]['features'],
        'RSS': rss,
        'Cp': cp,
        'AIC': mk.aic,
        'BIC': mk.bic,
        'Adj_R2': mk.rsquared_adj,
    })

crit_df = pd.DataFrame(rows)
print(crit_df[['k', 'RSS', 'Cp', 'AIC', 'BIC', 'Adj_R2']].round(2).to_string(index=False))

print(f"\nBest by Cp:     k={crit_df.loc[crit_df.Cp.idxmin(), 'k']}")
print(f"Best by AIC:    k={crit_df.loc[crit_df.AIC.idxmin(), 'k']}")
print(f"Best by BIC:    k={crit_df.loc[crit_df.BIC.idxmin(), 'k']}")
print(f"Best by Adj R²: k={crit_df.loc[crit_df.Adj_R2.idxmax(), 'k']}")

### 3.2 Plot all four criteria — 2×2 grid

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
criteria = [
    ('Cp',     True,  'steelblue'),
    ('AIC',    True,  'orange'),
    ('BIC',    True,  'green'),
    ('Adj_R2', False, 'tomato'),
]

for ax, (col, minimize, color) in zip(axes.flat, criteria):
    ax.plot(crit_df['k'], crit_df[col], 'o-', color=color, ms=8, lw=2)
    best_k = crit_df.loc[crit_df[col].idxmin() if minimize else crit_df[col].idxmax(), 'k']
    best_v = crit_df.loc[crit_df[col].idxmin() if minimize else crit_df[col].idxmax(), col]
    ax.scatter(best_k, best_v, s=150, marker='x', color='black', zorder=6,
               linewidths=2.5, label=f'best k={best_k}')
    ax.set_xlabel('Number of Predictors')
    ax.set_title(f'{col} ({"minimise" if minimize else "maximise"})')
    ax.legend()

plt.suptitle('Model Selection Criteria — Credit data (best-subset models)', fontsize=13)
plt.tight_layout()
plt.show()

print("\nKey insight:")
print("  BIC penalises with log(n) > 2 (for n > 7), so it tends to select smaller models.")
print("  Adj R² pays a price for adding unnecessary predictors (denominator n−d−1 grows).")

### 3.3 BIC vs AIC — heavier penalty comparison

In [ ]:
print(f"n = {n_cr}, log(n) = {np.log(n_cr):.2f}")
print(f"BIC penalty per parameter: log(n) = {np.log(n_cr):.2f}")
print(f"AIC/Cp penalty per parameter: 2")
print(f"")
print(f"For n={n_cr}: BIC penalises {np.log(n_cr)/2:.1f}× more than AIC per parameter.")
print("")
print("Rule of thumb:")
print("  AIC / Cp  → better for prediction (want low test MSE)")
print("  BIC       → better for model identification (want true model)")

---
## 4. Validation & Cross-Validation for Model Selection

Information criteria require σ̂² from the full model. **Cross-validation** provides a direct, model-agnostic estimate of test error — no σ̂² needed.

### 4.1 Validation set approach

In [ ]:
# 75/25 split
X_train, X_val, y_train, y_val = train_test_split(
    X_cr, y_cr, test_size=0.25, random_state=42
)

val_mse = {}
for k in range(1, p_cr + 1):
    idx = [num_cols.index(f) for f in best_subsets[k]['features']]
    lr = LinearRegression().fit(X_train[:, idx], y_train)
    preds = lr.predict(X_val[:, idx])
    val_mse[k] = mean_squared_error(y_val, preds)

print("Validation set MSE per model size:")
for k, mse in val_mse.items():
    marker = " ← minimum" if k == min(val_mse, key=val_mse.get) else ""
    print(f"  k={k}: {mse:.2f}{marker}")

### 4.2 10-fold cross-validation

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=0)
cv_mse_mean = {}
cv_mse_se   = {}

for k in range(1, p_cr + 1):
    idx = [num_cols.index(f) for f in best_subsets[k]['features']]
    Xk  = X_cr[:, idx]
    scores = -cross_val_score(
        LinearRegression(), Xk, y_cr, cv=kf,
        scoring='neg_mean_squared_error'
    )
    cv_mse_mean[k] = scores.mean()
    cv_mse_se[k]   = scores.std() / np.sqrt(10)

print("10-fold CV MSE per model size:")
best_k_cv = min(cv_mse_mean, key=cv_mse_mean.get)
for k in cv_mse_mean:
    marker = " ← minimum" if k == best_k_cv else ""
    print(f"  k={k}: {cv_mse_mean[k]:.2f} ± {cv_mse_se[k]:.2f}{marker}")

### 4.3 BIC vs. validation vs. CV — comparison plot

In [ ]:
ks = list(range(1, p_cr + 1))
bic_vals = crit_df['BIC'].values
# Normalise BIC to same scale as MSE for visual comparison
bic_norm = (bic_vals - bic_vals.min()) / (bic_vals.max() - bic_vals.min())
val_arr  = np.array([val_mse[k] for k in ks])
cv_arr   = np.array([cv_mse_mean[k] for k in ks])
cv_se    = np.array([cv_mse_se[k]   for k in ks])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# BIC
axes[0].plot(ks, bic_vals, 'go-', ms=8, lw=2)
axes[0].scatter(crit_df.loc[crit_df.BIC.idxmin(), 'k'],
                crit_df.BIC.min(), s=150, marker='x', color='k', lw=2.5, zorder=5)
axes[0].set_xlabel('# Predictors'); axes[0].set_title('BIC')

# Validation set
axes[1].plot(ks, val_arr, 'bs-', ms=8, lw=2)
best_val = min(val_mse, key=val_mse.get)
axes[1].scatter(best_val, val_mse[best_val], s=150, marker='x', color='k', lw=2.5, zorder=5)
axes[1].set_xlabel('# Predictors'); axes[1].set_title('Validation Set MSE')

# CV with error bars
axes[2].errorbar(ks, cv_arr, yerr=cv_se, fmt='ro-', ms=8, lw=2, capsize=5,
                 elinewidth=1, label='CV MSE ± 1 SE')
axes[2].scatter(best_k_cv, cv_mse_mean[best_k_cv], s=150, marker='x', color='k',
                lw=2.5, zorder=5, label='minimum')
# One-SE rule
threshold = cv_mse_mean[best_k_cv] + cv_mse_se[best_k_cv]
axes[2].axhline(threshold, color='purple', ls='--', lw=1.5, label='min + 1 SE')
one_se_k = min(k for k in ks if cv_mse_mean[k] <= threshold)
axes[2].axvline(one_se_k, color='purple', ls=':', lw=1.5, label=f'1-SE rule: k={one_se_k}')
axes[2].set_xlabel('# Predictors'); axes[2].set_title('10-fold CV MSE')
axes[2].legend(fontsize=8)

plt.suptitle('Comparing selection criteria — Credit data', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nBIC selects k={crit_df.loc[crit_df.BIC.idxmin(), 'k']}")
print(f"Validation set selects k={best_val}")
print(f"CV minimum selects k={best_k_cv}")
print(f"One-SE rule selects k={one_se_k} (simplest model within 1 SE of CV minimum)")

---
## 5. Ridge Regression

**Objective:** minimise  RSS + λ Σβⱼ²

The L2 penalty shrinks all coefficients toward zero but **never to exactly zero**. Key properties:
- Reduces variance at the cost of a small bias increase
- Requires standardised predictors (the penalty is not scale-equivariant)
- λ is selected by cross-validation

### 5.1 Standardise predictors — why it matters

In [ ]:
# Use diabetes dataset (sklearn standard)
scaler = StandardScaler()
Xs_diab = scaler.fit_transform(X_diab)   # zero mean, unit variance

print("Before scaling:")
print(pd.DataFrame(X_diab, columns=diab_feature_names).describe().round(2))

print("\nAfter scaling (all features have mean≈0, std≈1):")
print(pd.DataFrame(Xs_diab, columns=diab_feature_names).describe().round(3))

### 5.2 Ridge coefficient paths vs. log(λ)

In [ ]:
alphas = np.logspace(-3, 5, 300)   # sklearn uses 'alpha' for λ
ridge_coefs = np.array([
    Ridge(alpha=a).fit(Xs_diab, y_diab).coef_ for a in alphas
])

# Also compute OLS coefficients
ols_coefs = LinearRegression().fit(Xs_diab, y_diab).coef_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: coefficients vs log(α)
colors = plt.cm.tab10(np.linspace(0, 1, len(diab_feature_names)))
for i, (name, color) in enumerate(zip(diab_feature_names, colors)):
    axes[0].plot(np.log10(alphas), ridge_coefs[:, i], color=color, lw=2, label=name)
axes[0].axhline(0, color='k', lw=0.8, ls='--')
axes[0].axvline(0, color='grey', lw=0.5, ls=':')
axes[0].set_xlabel('log₁₀(λ)'); axes[0].set_ylabel('Standardised coefficient')
axes[0].set_title('Ridge: coefficient paths vs λ')
axes[0].legend(fontsize=8, ncol=2, loc='upper right')

# Right panel: coefficients vs ||β_ridge|| / ||β_OLS|| (like ISLR Figure 6.4)
norm_ols   = np.linalg.norm(ols_coefs)
norm_ridge = np.linalg.norm(ridge_coefs, axis=1)
ratio = norm_ridge / norm_ols
for i, (name, color) in enumerate(zip(diab_feature_names, colors)):
    axes[1].plot(ratio, ridge_coefs[:, i], color=color, lw=2, label=name)
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel('||β̂_ridge|| / ||β̂_OLS||'); axes[1].set_ylabel('Standardised coefficient')
axes[1].set_title('Ridge: coefficient paths vs shrinkage ratio')
axes[1].legend(fontsize=8, ncol=2)

plt.suptitle('Ridge Regression — Diabetes dataset', fontsize=13)
plt.tight_layout()
plt.show()

print("Observation: at ratio=1 (left edge) we have OLS; at ratio=0 all coefficients → 0.")
print("Ridge NEVER drives any coefficient to exactly zero.")

### 5.3 Bias–variance trade-off via simulation

In [ ]:
# Simulate n=50, p=45 with all non-zero coefficients (like ISLR Figure 6.5)
rng = np.random.default_rng(1)
n_sim, p_sim = 100, 45
X_sim = rng.normal(size=(n_sim, p_sim))
beta_true = rng.normal(0, 1, p_sim)   # all non-zero

# Generate test set
X_test_sim = rng.normal(size=(5000, p_sim))
y_test_sim = X_test_sim @ beta_true + rng.normal(0, 1, 5000)

n_repeats = 50
alphas_bv = np.logspace(-2, 3, 50)
preds_all = np.zeros((n_repeats, len(alphas_bv), 5000))

for rep in range(n_repeats):
    y_sim = X_sim @ beta_true + rng.normal(0, 1, n_sim)
    sc = StandardScaler().fit(X_sim)
    for ia, a in enumerate(alphas_bv):
        coef = Ridge(alpha=a).fit(sc.transform(X_sim), y_sim).coef_
        preds_all[rep, ia, :] = sc.transform(X_test_sim) @ coef

mean_pred = preds_all.mean(axis=0)   # shape: (n_alpha, 5000)
bias2 = ((mean_pred - y_test_sim) ** 2).mean(axis=1)
variance = preds_all.var(axis=0).mean(axis=1)
test_mse = bias2 + variance + 1.0    # +σ²=1

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.log10(alphas_bv), bias2,    color='black',  lw=2, label='Bias²')
ax.plot(np.log10(alphas_bv), variance, color='green',  lw=2, label='Variance')
ax.plot(np.log10(alphas_bv), test_mse, color='purple', lw=2.5, label='Test MSE')
ax.axhline(1.0, color='grey', ls='--', lw=1, label='Irreducible σ²=1')
opt = np.argmin(test_mse)
ax.axvline(np.log10(alphas_bv[opt]), color='purple', ls=':', lw=1.5,
           label=f'Optimal λ={alphas_bv[opt]:.2f}')
ax.scatter(np.log10(alphas_bv[opt]), test_mse[opt], s=120, color='purple',
           zorder=6, marker='x', linewidths=2.5)
ax.set_xlabel('log₁₀(λ)'); ax.set_ylabel('Error')
ax.set_title('Ridge: Bias–Variance trade-off (n=100, p=45)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"OLS (λ→0): Bias²={bias2[0]:.3f}, Var={variance[0]:.3f}, MSE={test_mse[0]:.3f}")
print(f"Optimal λ={alphas_bv[opt]:.2f}: Bias²={bias2[opt]:.3f}, Var={variance[opt]:.3f}, MSE={test_mse[opt]:.3f}")

### 5.4 Selecting λ with RidgeCV

In [ ]:
alpha_grid = np.logspace(-3, 5, 200)

# RidgeCV uses efficient LOO-CV (GCV) by default
rcv = RidgeCV(alphas=alpha_grid, cv=10).fit(Xs_diab, y_diab)

print(f"RidgeCV selected λ = {rcv.alpha_:.4f}")
print(f"All {len(rcv.coef_)} coefficients are non-zero (Ridge never zeroes out)")
print()

# Manual CV across grid to plot the curve
cv_mse_ridge = []
for a in alpha_grid:
    scores = -cross_val_score(
        Ridge(alpha=a), Xs_diab, y_diab, cv=10,
        scoring='neg_mean_squared_error'
    )
    cv_mse_ridge.append(scores.mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.log10(alpha_grid), cv_mse_ridge, color='steelblue', lw=2)
ax.axvline(np.log10(rcv.alpha_), color='red', ls='--', lw=1.5,
           label=f'CV-selected λ={rcv.alpha_:.3f}')
ax.set_xlabel('log₁₀(λ)'); ax.set_ylabel('CV MSE')
ax.set_title('RidgeCV: 10-fold CV error vs λ')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. The Lasso

**Objective:** minimise  RSS + λ Σ|βⱼ|

The L1 penalty forces some coefficients to be **exactly zero** — the lasso performs *variable selection*. The key difference from Ridge: the L1 constraint region is a diamond with corners on the axes, and the RSS ellipse tends to contact those corners.

### 6.1 Lasso coefficient paths

In [ ]:
lasso_alphas = np.logspace(-3, 1, 300)
lasso_coefs  = np.array([
    Lasso(alpha=a, max_iter=10000).fit(Xs_diab, y_diab).coef_
    for a in lasso_alphas
])

# Shrinkage ratio (L1 norm)
ols_l1 = np.sum(np.abs(ols_coefs))
lasso_l1 = np.sum(np.abs(lasso_coefs), axis=1)
ratio_l1 = lasso_l1 / ols_l1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: vs log(λ)
for i, (name, color) in enumerate(zip(diab_feature_names, colors)):
    axes[0].plot(np.log10(lasso_alphas), lasso_coefs[:, i], color=color, lw=2, label=name)
axes[0].axhline(0, color='k', lw=0.8, ls='--')
axes[0].set_xlabel('log₁₀(λ)'); axes[0].set_ylabel('Standardised coefficient')
axes[0].set_title('Lasso: coefficient paths vs λ')
axes[0].legend(fontsize=8, ncol=2)

# Right: vs L1 ratio
for i, (name, color) in enumerate(zip(diab_feature_names, colors)):
    axes[1].plot(ratio_l1, lasso_coefs[:, i], color=color, lw=2, label=name)
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel('||β̂_lasso||₁ / ||β̂_OLS||₁')
axes[1].set_ylabel('Standardised coefficient')
axes[1].set_title('Lasso: coefficient paths vs shrinkage ratio')
axes[1].legend(fontsize=8, ncol=2)

plt.suptitle('Lasso — Diabetes dataset', fontsize=13)
plt.tight_layout()
plt.show()

# Count how many features are zeroed at the CV-optimal λ (computed below)
lcv = LassoCV(cv=10, max_iter=20000, random_state=0).fit(Xs_diab, y_diab)
zeroed = [n for n, c in zip(diab_feature_names, lcv.coef_) if c == 0.0]
kept   = [n for n, c in zip(diab_feature_names, lcv.coef_) if c != 0.0]
print(f"LassoCV selected λ = {lcv.alpha_:.4f}")
print(f"Zeroed out: {zeroed}")
print(f"Kept: {kept}")

### 6.2 Geometric picture — diamond vs. circle

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# OLS minimum for this illustration
beta_ols_illus = np.array([1.8, 1.4])

for ax, title, constraint_pts in zip(
    axes,
    ['Lasso (L1 — diamond constraint)', 'Ridge (L2 — circle constraint)'],
    [
        np.column_stack([  # L1 diamond
            np.array([0, 1.2, 0, -1.2, 0]),
            np.array([1.2, 0, -1.2, 0, 1.2])
        ]),
        None
    ]
):
    # RSS ellipses
    for r in [0.8, 1.4, 2.2, 3.2]:
        ellipse_x = beta_ols_illus[0] + r * 0.9 * np.cos(theta)
        ellipse_y = beta_ols_illus[1] + r * 0.5 * np.sin(theta)
        ax.plot(ellipse_x, ellipse_y, color='#dc2626', lw=1.2, alpha=0.7)

    # OLS estimate
    ax.scatter(*beta_ols_illus, s=100, color='#dc2626', zorder=6)
    ax.text(beta_ols_illus[0] + 0.08, beta_ols_illus[1], 'β̂ (OLS)',
            color='#dc2626', fontsize=10)

    # Constraint region
    if constraint_pts is not None:   # Lasso diamond
        from matplotlib.patches import Polygon
        diamond = Polygon(constraint_pts, closed=True,
                          facecolor='#06b6d4', alpha=0.25, edgecolor='#06b6d4', lw=2)
        ax.add_patch(diamond)
        # Contact at corner (β₁=0 corner is at (−1.2, 0))
        ax.scatter(-1.2, 0, s=120, color='black', zorder=7)
        ax.annotate('contact\nat corner\n→ β₁=0', xy=(-1.2, 0),
                    xytext=(-2.2, 0.5),
                    arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
    else:   # Ridge circle
        circle = plt.Circle((0, 0), 1.2, color='#06b6d4', alpha=0.25,
                             fill=True, linewidth=2, edgecolor='#06b6d4')
        ax.add_patch(circle)
        # Contact on smooth arc
        cx, cy = -0.9, 0.8
        ax.scatter(cx, cy, s=120, color='black', zorder=7)
        ax.annotate('contact on\nsmooth arc\n→ β≠0', xy=(cx, cy),
                    xytext=(-2.5, 0.5),
                    arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlim(-2.8, 2.8); ax.set_ylim(-2.0, 2.5)
    ax.set_xlabel('β₁'); ax.set_ylabel('β₂')
    ax.set_title(title)
    ax.set_aspect('equal')

plt.suptitle('Why Lasso produces exact zeros (corners) but Ridge does not', fontsize=12)
plt.tight_layout()
plt.show()

### 6.3 Lasso vs. Ridge — CV error comparison

In [ ]:
# Lasso CV
lasso_cv_mse = []
for a in lasso_alphas:
    s = -cross_val_score(Lasso(alpha=a, max_iter=10000), Xs_diab, y_diab,
                         cv=10, scoring='neg_mean_squared_error')
    lasso_cv_mse.append(s.mean())

# Ridge CV (reuse from above)
ridge_cv_mse_comparable = []
for a in lasso_alphas:  # same alpha range for comparison
    s = -cross_val_score(Ridge(alpha=a), Xs_diab, y_diab,
                         cv=10, scoring='neg_mean_squared_error')
    ridge_cv_mse_comparable.append(s.mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.log10(lasso_alphas), lasso_cv_mse,
        color='steelblue', lw=2, label='Lasso CV MSE')
ax.plot(np.log10(lasso_alphas), ridge_cv_mse_comparable,
        color='tomato', lw=2, ls='--', label='Ridge CV MSE')

lasso_best_a = lasso_alphas[np.argmin(lasso_cv_mse)]
ridge_best_a = lasso_alphas[np.argmin(ridge_cv_mse_comparable)]
ax.axvline(np.log10(lasso_best_a), color='steelblue', ls=':', lw=1.2)
ax.axvline(np.log10(ridge_best_a), color='tomato', ls=':', lw=1.2)

ax.set_xlabel('log₁₀(λ)'); ax.set_ylabel('CV MSE')
ax.set_title('Lasso vs Ridge — 10-fold CV MSE on Diabetes data')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Lasso: best λ={lasso_best_a:.4f}, min CV-MSE={min(lasso_cv_mse):.2f}")
print(f"Ridge: best λ={ridge_best_a:.4f}, min CV-MSE={min(ridge_cv_mse_comparable):.2f}")

---
## 7. Dimension Reduction: PCR and PLS

Instead of shrinking coefficients, **transform** predictors into M < p new features, then regress Y on those M features.

- **PCR (Principal Components Regression):** unsupervised — uses PCA directions (maximum variance in X)
- **PLS (Partial Least Squares):** supervised — directions chosen to maximise covariance with Y

### 7.1 PCA: first two principal components

In [ ]:
pca_full = PCA().fit(Xs_diab)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
axes[0].bar(range(1, len(explained)+1), explained, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree plot — Diabetes data')

# Cumulative
axes[1].plot(range(1, len(cumulative)+1), cumulative, 'o-', color='steelblue', ms=7)
axes[1].axhline(0.9, color='red', ls='--', lw=1, label='90% threshold')
n90 = np.argmax(cumulative >= 0.9) + 1
axes[1].axvline(n90, color='red', ls=':', lw=1)
axes[1].set_xlabel('Number of PCs'); axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative variance explained')
axes[1].legend()

plt.suptitle('PCA on Diabetes data', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Number of PCs needed for 90% variance: {n90}")

### 7.2 PCR — regression on first M principal components

In [ ]:
Ms = range(1, len(diab_feature_names) + 1)

pcr_cv_mse  = []
for M in Ms:
    pipe = Pipeline([
        ('pca', PCA(n_components=M)),
        ('lr',  LinearRegression()),
    ])
    scores = -cross_val_score(pipe, Xs_diab, y_diab, cv=10,
                              scoring='neg_mean_squared_error')
    pcr_cv_mse.append(scores.mean())

best_M_pcr = int(np.argmin(pcr_cv_mse)) + 1
print(f"PCR: optimal M = {best_M_pcr}, CV-MSE = {pcr_cv_mse[best_M_pcr-1]:.2f}")

### 7.3 PLS — supervised component extraction

In [ ]:
pls_cv_mse = []
for M in Ms:
    pls = PLSRegression(n_components=M)
    scores = -cross_val_score(pls, Xs_diab, y_diab, cv=10,
                              scoring='neg_mean_squared_error')
    pls_cv_mse.append(scores.mean())

best_M_pls = int(np.argmin(pls_cv_mse)) + 1
print(f"PLS: optimal M = {best_M_pls}, CV-MSE = {pls_cv_mse[best_M_pls-1]:.2f}")

### 7.4 Compare all methods — OLS, Ridge, Lasso, PCR, PLS

In [ ]:
# CV MSE for each method (10-fold)
ols_mse = -cross_val_score(LinearRegression(), Xs_diab, y_diab, cv=10,
                            scoring='neg_mean_squared_error').mean()
ridge_best_mse = min(cv_mse_ridge)
lasso_best_mse = min(lasso_cv_mse)
pcr_best_mse   = min(pcr_cv_mse)
pls_best_mse   = min(pls_cv_mse)

# ── Plot: PCR vs PLS ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(Ms, pcr_cv_mse, 'b-o', ms=7, label='PCR (unsupervised PCA)')
ax.plot(Ms, pls_cv_mse, 'g--s', ms=7, label='PLS (supervised)')
ax.axhline(ols_mse, color='red',      ls=':',  lw=1.5, label=f'OLS ({ols_mse:.0f})')
ax.axhline(ridge_best_mse, color='orange',  ls='-.', lw=1.5, label=f'Ridge ({ridge_best_mse:.0f})')
ax.axhline(lasso_best_mse, color='purple',  ls='-.', lw=1.5, label=f'Lasso ({lasso_best_mse:.0f})')
ax.scatter(best_M_pcr, pcr_cv_mse[best_M_pcr-1], s=120, color='blue', zorder=6,
           marker='x', linewidths=2.5)
ax.scatter(best_M_pls, pls_cv_mse[best_M_pls-1], s=120, color='green', zorder=6,
           marker='x', linewidths=2.5)
ax.set_xlabel('Number of Components M')
ax.set_ylabel('10-fold CV MSE')
ax.set_title('PCR vs PLS — Diabetes dataset')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Summary table
summary = pd.DataFrame({
    'Method': ['OLS', 'Ridge', 'Lasso', 'PCR', 'PLS'],
    'CV MSE': [ols_mse, ridge_best_mse, lasso_best_mse, pcr_best_mse, pls_best_mse],
    'Variable selection': ['No', 'No', 'Yes (L1)', 'No', 'No'],
    'Key parameter': ['—', f'λ={rcv.alpha_:.3f}',
                      f'λ={lcv.alpha_:.4f}',
                      f'M={best_M_pcr}', f'M={best_M_pls}'],
})
print(summary.to_string(index=False))

---
## Exercises

Use the **Diabetes dataset** (already loaded as `X_diab`, `y_diab`, `Xs_diab`) for all four exercises.

### Exercise 1 — Best Subset + BIC selection

Tasks:
1. Run best subset selection over all 2¹⁰ subsets of the 10 Diabetes features.
2. For each best-of-size-k model, compute the **BIC** using `statsmodels`.
3. Plot BIC vs k and mark the selected model with an × symbol.
4. Which features are in the BIC-selected model? Print them.

In [ ]:
# Your code here


#### Exercise 1 — Solution

In [ ]:
p_d = Xs_diab.shape[1]

best_ex1 = {}
for k in range(1, p_d + 1):
    best_rss, best_combo = np.inf, None
    for combo in combinations(range(p_d), k):
        Xk = Xs_diab[:, combo]
        lr = LinearRegression().fit(Xk, y_diab)
        rss = np.sum((y_diab - lr.predict(Xk))**2)
        if rss < best_rss:
            best_rss, best_combo = rss, combo
    best_ex1[k] = (best_rss, best_combo)

bic_ex1 = []
for k, (rss, combo) in best_ex1.items():
    Xk = sm.add_constant(Xs_diab[:, combo])
    mk = sm.OLS(y_diab, Xk).fit()
    bic_ex1.append(mk.bic)

best_k_ex1 = int(np.argmin(bic_ex1)) + 1
best_features_ex1 = [diab_feature_names[i] for i in best_ex1[best_k_ex1][1]]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, p_d+1), bic_ex1, 'go-', ms=8, lw=2)
ax.scatter(best_k_ex1, bic_ex1[best_k_ex1-1], s=200, color='black',
           marker='x', zorder=6, linewidths=2.5, label=f'Selected k={best_k_ex1}')
ax.set_xlabel('# Predictors'); ax.set_ylabel('BIC')
ax.set_title('Best Subset BIC — Diabetes data'); ax.legend()
plt.tight_layout(); plt.show()

print(f"BIC-selected k = {best_k_ex1}")
print(f"Selected features: {best_features_ex1}")

### Exercise 2 — RidgeCV: coefficient paths + optimal α

Tasks:
1. Fit `RidgeCV` with `cv=10` on the standardised Diabetes data.
2. Plot coefficient paths vs log₁₀(α) for all 10 features.
3. Add a vertical dashed line at the CV-selected α.
4. Print the coefficient vector at the selected α. Are any exactly zero?

In [ ]:
# Your code here


#### Exercise 2 — Solution

In [ ]:
alphas_ex2 = np.logspace(-3, 5, 300)
coefs_ex2  = np.array([Ridge(alpha=a).fit(Xs_diab, y_diab).coef_ for a in alphas_ex2])
rcv_ex2    = RidgeCV(alphas=alphas_ex2, cv=10).fit(Xs_diab, y_diab)

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, color) in enumerate(zip(diab_feature_names, colors)):
    ax.plot(np.log10(alphas_ex2), coefs_ex2[:, i], color=color, lw=2, label=name)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.axvline(np.log10(rcv_ex2.alpha_), color='red', ls='--', lw=2,
           label=f'CV α={rcv_ex2.alpha_:.3f}')
ax.set_xlabel('log₁₀(α)'); ax.set_ylabel('Coefficient')
ax.set_title('RidgeCV — Diabetes data')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

print(f"CV-selected α = {rcv_ex2.alpha_:.4f}")
coef_df = pd.DataFrame({'feature': diab_feature_names, 'coef': rcv_ex2.coef_})
print(coef_df.to_string(index=False))
print(f"\nExact zeros: {(rcv_ex2.coef_ == 0).sum()} — Ridge never zeroes out features.")

### Exercise 3 — LassoCV: which features are zeroed out?

Tasks:
1. Fit `LassoCV` with `cv=10` on the standardised Diabetes data.
2. Plot the Lasso coefficient paths vs log₁₀(α), with a vertical line at the CV-selected α.
3. Report which features are exactly zeroed out and which are retained.
4. Compare: how many features does Lasso keep vs Ridge (Exercise 2)?

In [ ]:
# Your code here


#### Exercise 3 — Solution

In [ ]:
alphas_ex3  = np.logspace(-3, 1, 300)
coefs_ex3   = np.array([
    Lasso(alpha=a, max_iter=10000).fit(Xs_diab, y_diab).coef_
    for a in alphas_ex3
])
lcv_ex3 = LassoCV(cv=10, max_iter=20000, random_state=0).fit(Xs_diab, y_diab)

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, color) in enumerate(zip(diab_feature_names, colors)):
    ax.plot(np.log10(alphas_ex3), coefs_ex3[:, i], color=color, lw=2, label=name)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.axvline(np.log10(lcv_ex3.alpha_), color='red', ls='--', lw=2,
           label=f'CV α={lcv_ex3.alpha_:.4f}')
ax.set_xlabel('log₁₀(α)'); ax.set_ylabel('Coefficient')
ax.set_title('LassoCV — Diabetes data')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

zeroed_ex3 = [n for n, c in zip(diab_feature_names, lcv_ex3.coef_) if c == 0]
kept_ex3   = [n for n, c in zip(diab_feature_names, lcv_ex3.coef_) if c != 0]
print(f"LassoCV selected α = {lcv_ex3.alpha_:.4f}")
print(f"Zeroed out ({len(zeroed_ex3)} features): {zeroed_ex3}")
print(f"Kept ({len(kept_ex3)} features): {kept_ex3}")
print(f"\nRidge keeps all {p_d} features; Lasso keeps only {len(kept_ex3)}.")

### Exercise 4 — PCR vs PLS: CV-MSE vs M

Tasks:
1. For M = 1, 2, …, 10: compute 10-fold CV-MSE for PCR and PLS on Diabetes data.
2. Plot CV-MSE vs M for both methods on the same axes.
3. Add a horizontal dotted line for OLS CV-MSE.
4. Which method reaches its minimum with fewer components? Why?
5. Which components method beats OLS first?

In [ ]:
# Your code here


#### Exercise 4 — Solution

In [ ]:
Ms_ex4    = range(1, p_d + 1)
pcr_ex4   = []
pls_ex4   = []

for M in Ms_ex4:
    pipe_pcr = Pipeline([('pca', PCA(n_components=M)), ('lr', LinearRegression())])
    pcr_ex4.append(-cross_val_score(pipe_pcr, Xs_diab, y_diab, cv=10,
                                    scoring='neg_mean_squared_error').mean())
    pls_m = PLSRegression(n_components=M)
    pls_ex4.append(-cross_val_score(pls_m, Xs_diab, y_diab, cv=10,
                                    scoring='neg_mean_squared_error').mean())

ols_ex4 = -cross_val_score(LinearRegression(), Xs_diab, y_diab, cv=10,
                            scoring='neg_mean_squared_error').mean()

best_M_pcr_ex4 = int(np.argmin(pcr_ex4)) + 1
best_M_pls_ex4 = int(np.argmin(pls_ex4)) + 1

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(Ms_ex4, pcr_ex4, 'b-o', ms=7, lw=2, label=f'PCR (best M={best_M_pcr_ex4})')
ax.plot(Ms_ex4, pls_ex4, 'g--s', ms=7, lw=2, label=f'PLS (best M={best_M_pls_ex4})')
ax.axhline(ols_ex4, color='red', ls=':', lw=1.5, label=f'OLS ({ols_ex4:.0f})')
ax.scatter(best_M_pcr_ex4, pcr_ex4[best_M_pcr_ex4-1], s=150, color='blue',
           marker='x', lw=2.5, zorder=6)
ax.scatter(best_M_pls_ex4, pls_ex4[best_M_pls_ex4-1], s=150, color='green',
           marker='x', lw=2.5, zorder=6)
ax.set_xlabel('Number of Components M')
ax.set_ylabel('10-fold CV MSE')
ax.set_title('PCR vs PLS — Diabetes data')
ax.legend()
plt.tight_layout()
plt.show()

# First M that beats OLS
first_pcr = next((m for m, v in zip(Ms_ex4, pcr_ex4) if v < ols_ex4), None)
first_pls = next((m for m, v in zip(Ms_ex4, pls_ex4) if v < ols_ex4), None)

print(f"PCR best M={best_M_pcr_ex4}, CV-MSE={pcr_ex4[best_M_pcr_ex4-1]:.1f}")
print(f"PLS best M={best_M_pls_ex4}, CV-MSE={pls_ex4[best_M_pls_ex4-1]:.1f}")
print(f"OLS CV-MSE={ols_ex4:.1f}")
print(f"\nPLS first beats OLS at M={first_pls}; PCR at M={first_pcr}.")
print("PLS uses fewer components because its directions maximise covariance with Y.")